# Pull in CA Data

In [5]:
import pandas as pd
import os

# URL for the California School Directory file
url = "https://www.cde.ca.gov/schooldirectory/report?rid=dl1&tp=txt"

# Load the tab-delimited file
df_dir = pd.read_csv(
    url,
    sep="\t",          # tab-delimited
    dtype=str,         # keep IDs and codes as strings
    encoding="latin1"  # handle special characters safely
)

# Path to your raw data folder
output_path = r"C:\Users\ewpic\CA_AbsenteeismAnalysis\project\data\raw\california_school_directory.csv"

# Ensure the folder exists
os.makedirs(os.path.dirname(output_path), exist_ok=True)

# Save the file
df_dir.to_csv(output_path, index=False)

print("Saved to:", output_path)

Saved to: C:\Users\ewpic\CA_AbsenteeismAnalysis\project\data\raw\california_school_directory.csv


In [6]:
import pandas as pd
import os

# URL for the California Chronic Absenteeism 2025 file
url = "https://www3.cde.ca.gov/demo-downloads/attendance/chronicabsenteeism25.txt"

# Load the tab-delimited file
df_abs = pd.read_csv(
    url,
    sep="\t",          # tab-delimited
    dtype=str,         # keep IDs and codes as strings
    encoding="latin1"  # handle special characters safely
)

# Path to your raw data folder
output_path = r"C:\Users\ewpic\CA_AbsenteeismAnalysis\project\data\raw\california_chronic_absenteeism_2025.csv"

# Ensure the folder exists
os.makedirs(os.path.dirname(output_path), exist_ok=True)

# Save the file
df_abs.to_csv(output_path, index=False)

print("Saved to:", output_path)


Saved to: C:\Users\ewpic\CA_AbsenteeismAnalysis\project\data\raw\california_chronic_absenteeism_2025.csv


In [4]:
import pandas as pd
import os

# URL for the California Enrollment (2024–25) file
url = "https://www3.cde.ca.gov/demo-downloads/census/cdenroll2425.txt"

# Load the tab-delimited file
df_enr = pd.read_csv(
    url,
    sep="\t",
    dtype=str,
    encoding="latin1",
    engine="python"     # python engine handles large tab files better
)

# Path to your raw data folder
output_path = r"C:\Users\ewpic\CA_AbsenteeismAnalysis\project\data\raw\california_enrollment_2024_25.csv"

# Ensure the folder exists
os.makedirs(os.path.dirname(output_path), exist_ok=True)

# Save the file
df_enr.to_csv(output_path, index=False)

print("Saved to:", output_path)

Saved to: C:\Users\ewpic\CA_AbsenteeismAnalysis\project\data\raw\california_enrollment_2024_25.csv


# Pull together data table with calculated feature columns

In [3]:
import duckdb

base = r"C:\Users\ewpic\CA_AbsenteeismAnalysis\project\data\raw"

df_enroll_q = duckdb.query(f"""

WITH
enr_raw AS (
    SELECT *
    FROM read_csv_auto('{base}\\california_enrollment_2024_25.csv')
),

dir_raw AS (
    SELECT *
    FROM read_csv_auto('{base}\\california_school_directory.csv')
),

abs_raw AS (
    SELECT *
    FROM read_csv(
        '{base}\\california_chronic_absenteeism_2025.csv',
        delim=',',
        quote='"',
        escape='"',
        header=true,
        ignore_errors=true
    )
),


enr AS (
    SELECT
        schoolcode,
        schoolname,
        districtcode,
        districtname,
        countyname,
        reportingcategory,
        total_enr
    FROM enr_raw
    WHERE aggregatelevel = 'S'
      AND charter = 'N'
      AND reportingcategory IN ('TA','SG_SD','SG_DS')
),

enr_pivot AS (
    SELECT *
    FROM enr
    PIVOT (
        SUM(CAST(total_enr AS DOUBLE))
        FOR reportingcategory IN ('TA','SG_SD','SG_DS')
    )
),

dir AS (
    SELECT
        CDScode,
        RIGHT(cdscode, 7) AS schoolcode,
        district,
        school,
        gsserved,
        virtual,
        SOC
    FROM dir_raw
),

abs AS (
    SELECT *
    FROM abs_raw
    WHERE "reporting category" = 'TA'
      AND "aggregate level" = 'S'
      AND ChronicAbsenteeismCount != '*'
),

d AS (
    SELECT
        b.schoolcode,
        b.schoolname,
        b.districtcode,
        b.countyname,
        c.district,
        c.gsserved,
        c.virtual,
        c.SOC,
        CASE WHEN c.virtual = 'V' THEN 1 ELSE 0 END AS VirtStat,
        COALESCE(b."TA", 0) AS TA,
        COALESCE(b."SG_SD", 0) AS SG_SD,
        COALESCE(b."SG_DS", 0) AS SG_DS
    FROM enr_pivot b
    LEFT JOIN dir c
        ON b.schoolcode = c.schoolcode
),

joined AS (
    SELECT
        d.*,
        e.ChronicAbsenteeismCount,
        CAST(e.ChronicAbsenteeismRate AS DOUBLE) AS chronicabsenteeismrate,
        COUNT(d.schoolcode) OVER (PARTITION BY d.countyname) AS Cnty_Sch_Cnt,
        NTILE(10) OVER (ORDER BY CAST(d.TA AS DOUBLE)) AS ENR_decile,
        NTILE(4) OVER (ORDER BY CAST(d.TA AS DOUBLE)) AS ENR_quartile,
        (CAST(d.SG_DS AS DOUBLE) / CAST(d.TA AS DOUBLE)) * 100 AS Rate_Sped,
        (CAST(d.SG_SD AS DOUBLE) / CAST(d.TA AS DOUBLE)) * 100 AS Rate_LI
    FROM d
    LEFT JOIN abs e
        ON d.schoolcode = e."school code"
    WHERE d.schoolcode NOT IN ('0000000','0000001')
      AND d.SOC IN ('60','61','62','64','65','66','67')
)

SELECT *
FROM joined
WHERE Cnty_Sch_Cnt >= 5

""").to_df()

df_enroll_q.head()

#save to processed data folder

import os

processed_path = r"C:\Users\ewpic\CA_AbsenteeismAnalysis\project\data\processed"

# Ensure the folder exists
os.makedirs(processed_path, exist_ok=True)

output_file = os.path.join(processed_path, "enrollment_absenteeism_merged.csv")

df_enroll_q.to_csv(output_file, index=False)

print("Saved processed file to:", output_file)

Saved processed file to: C:\Users\ewpic\CA_AbsenteeismAnalysis\project\data\processed\enrollment_absenteeism_merged.csv
